## **CUDA Runtime**

- Copies the data from the host to the device

- Load GPU kernel and execute it

- Copy the result from the device to the host

<hr>

## **Device Vs Host Naming Scheme**

- `h_Var` for host variable

- `d_Var` for device variable

**`__global__`**

-  It is a CUDA C/C++ keyword used to declare a function as a kernel function that can be executed on the GPU.

- A kernel function is a function that is executed on the GPU and can be called from the host (CPU) code.

**`__device__`**

- It is a CUDA C/C++ keyword used to declare a function or variable as a device function or variable that can only be accessed and executed on the GPU.

- A device function is a function that can only be called from other device functions or kernel functions, and cannot be called from host code.

**`__host__`**

- It is a CUDA C/C++ keyword used to declare a function as a host function that can only be executed on the CPU.

- A host function is a function that can only be called from other host functions or kernel functions, and cannot be called from device code.

<hr>

## **Memory Management**

**`cudaMalloc`**

- It is a CUDA C/C++ function used to allocate memory on the GPU device.

- The syntax for `cudaMalloc` is as follows:

```cpp
cudaError_t cudaMalloc(void **devPtr, size_t size);
```

**`cudaMemcpy`**

- It is a CUDA C/C++ function used to copy data between device -> device (One GPU location to another), host -> device, and device -> host.

- The syntax for `cudaMemcpy` is as follows:

```cpp
cudaError_t cudaMemcpy(void *dst, const void *src, size_t count, cudaMemcpyKind kind);
```

**`cudaFree`**

- It is a CUDA C/C++ function used to free memory that was previously allocated on the GPU device using `cudaMalloc`.

- The syntax for `cudaFree` is as follows:

```cpp
cudaError_t cudaFree(void *devPtr);
```

<hr>

What you have just outlined is the complete **Compilation Pipeline** of CUDA. This is one of the most brilliant engineering feats by NVIDIA, and understanding it explains exactly why CUDA dominates the high-performance computing world.

To understand this, you must first know one secret: **`nvcc` is not actually a compiler.** 

`nvcc` (NVIDIA CUDA Compiler) is a **Compiler Driver**. It is an orchestrator. When you feed it your `vadd.cu` file, `nvcc` takes out a scalpel, cuts your code perfectly in half (CPU vs. GPU), and sends each half down a completely different pipeline. 

Here is the in-depth breakdown of exactly what happens.

---

### Phase 1: The Host Code (CPU) Pipeline

Your CPU (`g++` / Ryzen processor) and your GPU (`nvcc` / RTX 3060) speak completely different languages. A standard C++ compiler like `g++` has absolutely no idea what `__global__` or `<<<1, 256>>>` means. If it sees them, it throws a syntax error.

**1. "Modified to run kernels"**
Before passing the code to your CPU compiler, `nvcc` performs a translation. It strips out all the GPU kernel code. Then, it looks at your kernel launch:
`vectorAdd<<<NUM_BLOCKS, NUM_THREADS>>>(d_a, d_b, d_c, n);`

It rewrites this weird `<<< >>>` syntax into standard, ugly C++ API functions. It transforms it into something like:
`cudaLaunchKernel((void*)vectorAdd, NUM_BLOCKS, NUM_THREADS, args);`

**2. "Compiled to x86 binary"**
Now that the file is 100% standard, pure C++, `nvcc` hands it over to your system's host compiler (in your case, `g++ 16.1`). 
`g++` compiles this code into standard **x86_64 machine code** that your AMD Ryzen 5 CPU can execute natively.

---

### Phase 2: The Device Code (GPU) Pipeline

While `g++` is busy compiling the CPU code, `nvcc` takes the `__global__` functions and starts compiling the GPU code. But it doesn't compile it to raw 1s and 0s immediately. 

**1. "Compiled to PTX"**
Instead of creating actual machine code, `nvcc` compiles your GPU code into **PTX (Parallel Thread Execution)**. 

PTX is an "Intermediate Representation." It is a fake, virtual assembly language. It acts as if it is compiling for a "perfect, theoretical" NVIDIA GPU with infinite registers and infinite memory. It is plain text, and you can actually read it if you want to!

**2. "Stable across multiple GPU generations"**
Why a fake assembly language? Because GPU hardware changes violently every 2 years. 
The physical micro-architecture of a 2016 GTX 1060 (Pascal) is wildly different from your 2021 RTX 3060 (Ampere). If NVIDIA forced you to write specific code for every single graphics card, developers would quit. 
PTX solves this. PTX is a universal, stable language. NVIDIA ensures that PTX generated 10 years ago is completely understandable by the NVIDIA driver of today.

---

### Phase 3: Execution and JIT (Just-In-Time) Compilation

So, you have your final executable file (like `./vadd`). Inside this executable is your compiled CPU code, and the text-based PTX blueprints for your GPU code. 

You press `Enter` to run the program.

**1. "PTX into native GPU instructions"**
Your physical RTX 3060 cannot read PTX. It only understands raw, physical microcode called **SASS** (Streaming ASSembler) or **cubin** (CUDA Binary). 

When your program reaches the `cudaLaunchKernel` step, the NVIDIA Display Driver on your computer wakes up. The driver looks at the PTX blueprints hidden in the executable, looks at your physical hardware, and says: *"Ah! You have an Ampere RTX 3060!"*

In a fraction of a millisecond, the driver performs **JIT (Just-In-Time) Compilation**. It translates the generic PTX text into the exact, highly-optimized, physical SASS machine code tailored perfectly for your specific GPU's silicon, and feeds it to the graphics card.

**2. "Allows for forward compatibility"**
This is the ultimate superpower of CUDA. 

Imagine you write your `vadd.cu` code today, compile it to an executable, and upload it to the internet.
Five years from now, someone downloads your executable and tries to run it on an **RTX 6090** (an architecture that hasn't even been invented yet). 

* If you had compiled it directly to RTX 3060 microcode, the program would crash. The RTX 6090 wouldn't understand the old instructions.
* **But because you embedded PTX**, the program runs perfectly. The future RTX 6090 driver will look at the PTX, JIT-compile it into RTX 6090 SASS, and run it flawlessly. 

You wrote code that runs on hardware from the future.

### Summary / How this affects your compilation flags:
Remember earlier when I told you to use `nvcc -arch=sm_86`? 
* `sm_86` is the physical architecture of your RTX 3060. 
* By using that flag, you told `nvcc`: *"Skip the JIT part for my machine! Compile it directly to physical SASS microcode for the RTX 3060 so it launches instantly!"* (This is called Ahead-Of-Time or AOT compilation).

If you were building software to sell on Steam to millions of gamers with different GPUs, you would use flags that embed the generic PTX so their individual drivers could JIT-compile it!